# Assignment 1 — Build an Optimal Deep Neural Network
**YunSeok Choi** | ys.choi@skku.edu | Sungkyunkwan University

## 0. Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1. Load Fashion-MNIST Dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

CLASS_NAMES = ['T-Shirt/Top','Trouser','Pullover','Dress','Coat',
               'Sandals','Shirt','Sneaker','Bag','Ankle Boots']

print(f'Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}')

In [ ]:
# Quick visualisation
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(CLASS_NAMES[label], fontsize=9)
    ax.axis('off')
plt.suptitle('Fashion-MNIST Samples', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Model Definition — Dense Network (MLP)

In [ ]:
class DenseNet(nn.Module):
    """
    Fully-connected (Dense) network with configurable hidden layers.
    Only Dense (Linear) layers are used — no CNN / ViT / RNN.
    """
    def __init__(self, hidden_sizes, activation=nn.ReLU,
                 use_batchnorm=False, dropout_rate=0.0):
        """
        Args:
            hidden_sizes  : list of ints, e.g. [256] / [256,128] / [256,128,64]
            activation    : activation class (nn.ReLU, nn.Tanh, …)
            use_batchnorm : whether to add BatchNorm after each hidden layer
            dropout_rate  : dropout probability (0 = disabled)
        """
        super().__init__()
        layers = [nn.Flatten()]                          # 28*28 → 784
        in_features = 784
        for out_features in hidden_sizes:
            layers.append(nn.Linear(in_features, out_features))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(out_features))
            layers.append(activation())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            in_features = out_features
        layers.append(nn.Linear(in_features, 10))       # output layer
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

## 3. Training & Evaluation Utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct = 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out  = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
        correct    += (out.argmax(1) == y).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        out  = model(X)
        total_loss += criterion(out, y).item() * X.size(0)
        correct    += (out.argmax(1) == y).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


def run_experiment(config, epochs=20):
    """Train a model from config dict and return history."""
    model     = DenseNet(**config['model_kwargs']).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = config['optimizer'](model.parameters(), **config['optim_kwargs'])
    scheduler = config.get('scheduler', None)
    if scheduler:
        scheduler = scheduler(optimizer, **config.get('sched_kwargs', {}))

    history = {'train_loss':[], 'train_acc':[], 'test_loss':[], 'test_acc':[]}
    for epoch in range(1, epochs + 1):
        tl, ta = train_one_epoch(model, train_loader, criterion, optimizer)
        vl, va = evaluate(model, test_loader, criterion)
        if scheduler:
            scheduler.step()
        history['train_loss'].append(tl)
        history['train_acc'].append(ta)
        history['test_loss'].append(vl)
        history['test_acc'].append(va)
        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch:02d} | Train Acc {ta:.4f} | Test Acc {va:.4f} | Test Loss {vl:.4f}')
    return model, history

## 4. Experiment 1 — Baseline: 1 / 2 / 3 Hidden Layers

> **Goal**: measure how depth alone affects accuracy (no extra tricks).

In [ ]:
depth_configs = {
    '1-hidden [256]':       {'hidden_sizes': [256]},
    '2-hidden [256,128]':   {'hidden_sizes': [256, 128]},
    '3-hidden [256,128,64]':{'hidden_sizes': [256, 128, 64]},
}

depth_results = {}
for name, kwargs in depth_configs.items():
    print(f'\n=== {name} ===')
    cfg = {
        'model_kwargs': kwargs,
        'optimizer':    optim.SGD,
        'optim_kwargs': {'lr': 0.01, 'momentum': 0.9},
    }
    _, hist = run_experiment(cfg, epochs=20)
    depth_results[name] = hist

In [ ]:
# Plot accuracy curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, hist in depth_results.items():
    axes[0].plot(hist['train_acc'], label=name)
    axes[1].plot(hist['test_acc'],  label=name)
for ax, title in zip(axes, ['Train Accuracy', 'Test Accuracy']):
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(fontsize=8); ax.grid(True)
plt.suptitle('Experiment 1: Depth Comparison (Baseline)')
plt.tight_layout()
plt.show()

# Summary table
print('\n--- Best Test Accuracy ---')
for name, hist in depth_results.items():
    print(f'  {name}: {max(hist["test_acc"]):.4f}')

## 5. Experiment 2 — Optimizer Comparison

> **Goal**: compare SGD, Adam, and RMSProp on the 3-hidden-layer model.

In [ ]:
BASE_HIDDEN = [256, 128, 64]

optim_configs = {
    'SGD (lr=0.01, momentum=0.9)': {
        'optimizer':    optim.SGD,
        'optim_kwargs': {'lr': 0.01, 'momentum': 0.9},
    },
    'Adam (lr=1e-3)': {
        'optimizer':    optim.Adam,
        'optim_kwargs': {'lr': 1e-3},
    },
    'RMSProp (lr=1e-3)': {
        'optimizer':    optim.RMSprop,
        'optim_kwargs': {'lr': 1e-3},
    },
}

optim_results = {}
for name, cfg in optim_configs.items():
    print(f'\n=== {name} ===')
    cfg['model_kwargs'] = {'hidden_sizes': BASE_HIDDEN}
    _, hist = run_experiment(cfg, epochs=20)
    optim_results[name] = hist

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, hist in optim_results.items():
    axes[0].plot(hist['train_loss'], label=name)
    axes[1].plot(hist['test_acc'],   label=name)
for ax, title in zip(axes, ['Train Loss', 'Test Accuracy']):
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(fontsize=8); ax.grid(True)
plt.suptitle('Experiment 2: Optimizer Comparison')
plt.tight_layout()
plt.show()

print('\n--- Best Test Accuracy ---')
for name, hist in optim_results.items():
    print(f'  {name}: {max(hist["test_acc"]):.4f}')

## 6. Experiment 3 — Regularization Techniques

> **Goal**: measure the effect of BatchNorm and Dropout.

In [ ]:
reg_configs = {
    'Baseline': {
        'model_kwargs': {'hidden_sizes': BASE_HIDDEN},
    },
    'BatchNorm': {
        'model_kwargs': {'hidden_sizes': BASE_HIDDEN, 'use_batchnorm': True},
    },
    'Dropout (p=0.3)': {
        'model_kwargs': {'hidden_sizes': BASE_HIDDEN, 'dropout_rate': 0.3},
    },
    'BatchNorm + Dropout': {
        'model_kwargs': {'hidden_sizes': BASE_HIDDEN, 'use_batchnorm': True, 'dropout_rate': 0.3},
    },
}

reg_results = {}
for name, cfg in reg_configs.items():
    print(f'\n=== {name} ===')
    cfg['optimizer']    = optim.Adam
    cfg['optim_kwargs'] = {'lr': 1e-3}
    _, hist = run_experiment(cfg, epochs=20)
    reg_results[name] = hist

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, hist in reg_results.items():
    axes[0].plot(hist['train_acc'], label=name)
    axes[1].plot(hist['test_acc'],  label=name)
for ax, title in zip(axes, ['Train Accuracy', 'Test Accuracy']):
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(fontsize=8); ax.grid(True)
plt.suptitle('Experiment 3: Regularization Techniques')
plt.tight_layout()
plt.show()

print('\n--- Best Test Accuracy ---')
for name, hist in reg_results.items():
    print(f'  {name}: {max(hist["test_acc"]):.4f}')

## 7. Best Model — Final Evaluation

Combine the best choices found above into a single final model.

In [ ]:
# TODO: fill in your best settings after reviewing experiments above
BEST_CONFIG = {
    'model_kwargs': {
        'hidden_sizes':  [256, 128, 64],
        'use_batchnorm': True,
        'dropout_rate':  0.3,
    },
    'optimizer':    optim.Adam,
    'optim_kwargs': {'lr': 1e-3},
    # Optional: LR scheduler
    # 'scheduler':    optim.lr_scheduler.StepLR,
    # 'sched_kwargs': {'step_size': 10, 'gamma': 0.5},
}

print('Training best model for 30 epochs…')
best_model, best_hist = run_experiment(BEST_CONFIG, epochs=30)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(best_hist['train_loss'], label='Train')
axes[0].plot(best_hist['test_loss'],  label='Test')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(best_hist['train_acc'], label='Train')
axes[1].plot(best_hist['test_acc'],  label='Test')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True)

plt.suptitle('Best Model — Training Curves')
plt.tight_layout()
plt.show()

print(f'\nFinal Test Accuracy: {best_hist["test_acc"][-1]:.4f}')